In [16]:
from datasets import Dataset
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [1]:
from google.colab import files
uploaded = files.upload()

Saving data.tar.gz to data.tar.gz


In [2]:
!tar -xzvf data.tar.gz

._data
data/
data/claims_dev.jsonl
data/cross_validation/
data/claims_train.jsonl
data/claims_test.jsonl
data/corpus.jsonl
data/cross_validation/._fold_4
data/cross_validation/fold_4/
data/cross_validation/fold_3/
data/cross_validation/fold_2/
data/cross_validation/fold_5/
data/cross_validation/._fold_1
data/cross_validation/fold_1/
data/cross_validation/fold_1/claims_dev_1.jsonl
data/cross_validation/fold_1/claims_train_1.jsonl
data/cross_validation/fold_5/claims_train_5.jsonl
data/cross_validation/fold_5/claims_dev_5.jsonl
data/cross_validation/fold_2/claims_dev_2.jsonl
data/cross_validation/fold_2/claims_train_2.jsonl
data/cross_validation/fold_3/claims_train_3.jsonl
data/cross_validation/fold_3/claims_dev_3.jsonl
data/cross_validation/fold_4/claims_dev_4.jsonl
data/cross_validation/fold_4/claims_train_4.jsonl


In [3]:
!find . -name "*.jsonl"

./data/corpus.jsonl
./data/claims_dev.jsonl
./data/cross_validation/fold_3/claims_dev_3.jsonl
./data/cross_validation/fold_3/claims_train_3.jsonl
./data/cross_validation/fold_5/claims_dev_5.jsonl
./data/cross_validation/fold_5/claims_train_5.jsonl
./data/cross_validation/fold_1/claims_dev_1.jsonl
./data/cross_validation/fold_1/claims_train_1.jsonl
./data/cross_validation/fold_2/claims_dev_2.jsonl
./data/cross_validation/fold_2/claims_train_2.jsonl
./data/cross_validation/fold_4/claims_train_4.jsonl
./data/cross_validation/fold_4/claims_dev_4.jsonl
./data/claims_train.jsonl
./data/claims_test.jsonl


In [5]:
import json

def load_corpus(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def load_claims(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read().strip()
    return [json.loads(l) for l in content.splitlines() if l.strip()]

corpus = load_corpus("data/corpus.jsonl")
claims_train = load_claims("data/claims_train.jsonl")
claims_dev   = load_claims("data/claims_dev.jsonl")
claims_test  = load_claims("data/claims_test.jsonl")
corpus_by_id = {p["doc_id"]: p for p in corpus}

print(f"Corpus papers: {len(corpus)}")
print(f"Train claims: {len(claims_train)}")

Corpus papers: 5183
Train claims: 809


In [6]:
def build_nli_examples(claims, corpus_by_id):
    examples = []
    for claim in claims:
        claim_text = claim["claim"]
        evidence = claim.get("evidence", {})

        if not evidence:
            examples.append({
                "claim": claim_text,
                "evidence_text": "",
                "label": "NOT_ENOUGH_INFO"
            })
            continue

        for doc_id_str, ev_list in evidence.items():
            doc_id = int(doc_id_str)
            paper = corpus_by_id.get(doc_id)
            if paper is None:
                continue
            for ev in ev_list:
                sent_ids = ev["sentences"]
                evidence_text = " ".join([paper["abstract"][i] for i in sent_ids])
                label = ev["label"]
                examples.append({
                    "claim": claim_text,
                    "evidence_text": evidence_text,
                    "label": label
                })
    return examples

train_examples = build_nli_examples(claims_train, corpus_by_id)
dev_examples = build_nli_examples(claims_dev, corpus_by_id)

print(f"Train examples: {len(train_examples)}")
print(f"Dev examples: {len(dev_examples)}")
print(train_examples[1])

Train examples: 1261
Dev examples: 450
{'claim': '1 in 5 million in UK have abnormal PrP positivity.', 'evidence_text': 'RESULTS Of the 32,441 appendix samples 16 were positive for abnormal PrP, indicating an overall prevalence of 493 per million population (95% confidence interval 282 to 801 per million).', 'label': 'CONTRADICT'}


In [29]:
label2id = {"SUPPORT": 0, "CONTRADICT": 1, "NOT_ENOUGH_INFO": 2}
id2label = {v: k for k, v in label2id.items()}
for ex in train_examples:
    ex["label_id"] = label2id[ex["label"]]
for ex in dev_examples:
    ex["label_id"] = label2id[ex["label"]]

In [30]:
def tokenize_examples(examples, tokenizer, max_length=256):
    claims = [ex["claim"] for ex in examples]
    evidences = [ex["evidence_text"] for ex in examples]
    labels = [ex["label_id"] for ex in examples]

    encodings = tokenizer(
        claims,
        evidences,
        truncation=True,
        padding=True,
        max_length=max_length
    )
    return encodings, labels

In [31]:
train_encodings, train_labels = tokenize_examples(train_examples, tokenizer)
dev_encodings, dev_labels = tokenize_examples(dev_examples, tokenizer)

In [32]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels
})
dev_dataset = Dataset.from_dict({
    "input_ids": dev_encodings["input_ids"],
    "attention_mask": dev_encodings["attention_mask"],
    "labels": dev_labels
})
print("Train dataset size:", len(train_dataset))
print("Dev dataset size:", len(dev_dataset))

Train dataset size: 1261
Dev dataset size: 450


In [7]:
!pip install transformers -q

In [47]:
from transformers import AutoModelForSequenceClassification
model_name = "allenai/scibert_scivocab_uncased"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
for name, param in model.named_parameters():
    print(name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allenai/scibert_scivocab_uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

In [48]:
for param in model.parameters():
    param.requires_grad = False
for name, param in model.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name or "pooler" in name or "classifier" in name:
        param.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

Trainable parameters: 14,768,643 / 109,920,771


In [49]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results_partial",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=10,
)

In [50]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
)

In [51]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.569054,0.640950,0.686667,0.596430
2,0.557946,0.582777,0.617778,0.613951
3,0.480664,0.545759,0.677778,0.685743
4,0.333914,0.554865,0.711111,0.698379
5,0.349369,0.573008,0.700000,0.701574
6,0.309837,0.610084,0.700000,0.704866
7,0.324741,0.618811,0.691111,0.704724
8,0.256257,0.630523,0.695556,0.703729


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=632, training_loss=0.4215716928055015, metrics={'train_runtime': 400.3603, 'train_samples_per_second': 25.197, 'train_steps_per_second': 1.579, 'total_flos': 1285670826550656.0, 'train_loss': 0.4215716928055015, 'epoch': 8.0})

In [52]:
model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

In [53]:
!zip -r final_model.zip final_model
from google.colab import files
files.download("final_model.zip")

  adding: final_model/ (stored 0%)
  adding: final_model/config.json (deflated 54%)
  adding: final_model/tokenizer_config.json (deflated 43%)
  adding: final_model/model.safetensors (deflated 7%)
  adding: final_model/tokenizer.json (deflated 71%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
test_examples = build_nli_examples(claims_test, corpus_by_id)
for ex in test_examples:
    ex["label_id"] = label2id[ex["label"]]
print(f"Test examples: {len(test_examples)}")

Test examples: 300


In [57]:
test_encodings, test_labels = tokenize_examples(test_examples, tokenizer)
from datasets import Dataset
test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"],
    "attention_mask": test_encodings["attention_mask"],
    "labels": test_labels
})

In [58]:
test_results = trainer.evaluate(test_dataset)
print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.256257,0.063884,8,0.976667,0.329399


{'eval_loss': 0.06388389319181442, 'eval_accuracy': 0.9766666666666667, 'eval_f1_macro': 0.32939853850477796}


In [59]:
from collections import Counter
print(Counter([ex["label"] for ex in test_examples]))

Counter({'NOT_ENOUGH_INFO': 300})


In [60]:
dev_results = trainer.evaluate(dev_dataset)
print(dev_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.256257,0.610084,8,0.700000,0.704866


{'eval_loss': 0.610083818435669, 'eval_accuracy': 0.7, 'eval_f1_macro': 0.7048660362490149}


In [63]:
import torch

def predict_verdict(claim_text, evidence_text, model=model, tokenizer=tokenizer, id2label=id2label):
    inputs = tokenizer(claim_text, evidence_text, truncation=True, padding=True, max_length=256, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = torch.argmax(probs).item()

    return {
        "verdict": id2label[pred_id],
        "confidence": float(probs[pred_id]),
        "all_probs": {id2label[i]: float(probs[i]) for i in range(len(probs))}
    }

In [64]:
example = dev_examples[1]
result = predict_verdict(example["claim"], example["evidence_text"])
print("Claim:", example["claim"])
print("Evidence:", example["evidence_text"])
print("True label:", example["label"])
print("Predicted:", result)

Claim: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
Evidence: We propose as an alternative explanation that variants much less common than the associated one may create "synthetic associations" by occurring, stochastically, more often in association with one of the alleles at the common site versus the other allele. We show that they are not only possible, but inevitable, and that under simple but reasonable genetic models, they are likely to account for or contribute to many of the recently identified signals reported in genome-wide association studies.
True label: SUPPORT
Predicted: {'verdict': 'SUPPORT', 'confidence': 0.9279332160949707, 'all_probs': {'SUPPORT': 0.9279332160949707, 'CONTRADICT': 0.05888380482792854, 'NOT_ENOUGH_INFO': 0.013183069415390491}}


In [65]:
from google.colab import files
uploaded = files.upload()

Saving rationale_selector.joblib to rationale_selector.joblib


In [67]:
import joblib
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

rationale_data = joblib.load("rationale_selector.joblib")
clf_evidence = rationale_data["model"]
vectorizer = rationale_data["vectorizer"]

def word_overlap(a, b):
    wa, wb = set(a.lower().split()), set(b.lower().split())
    if not wa or not wb:
        return 0.0
    return len(wa & wb) / len(wa | wb)

def featurize(rows):
    claim_vecs = vectorizer.transform([r["claim"] for r in rows])
    sent_vecs  = vectorizer.transform([r["sentence"] for r in rows])
    sims = np.array([cosine_similarity(claim_vecs[i], sent_vecs[i])[0][0] for i in range(len(rows))])
    overlaps = np.array([word_overlap(r["claim"], r["sentence"]) for r in rows])
    positions = np.array([r["sentence_idx"] for r in rows])
    lengths = np.array([len(r["sentence"].split()) for r in rows])
    X = np.column_stack([sims, overlaps, positions, lengths])
    return X

def select_evidence(claim_text, retrieved_paper, clf=clf_evidence, threshold=0.5):
    abstract = retrieved_paper["abstract"]
    rows = [{"claim": claim_text, "sentence": s, "sentence_idx": i} for i, s in enumerate(abstract)]
    X = featurize(rows)
    probs = clf.predict_proba(X)[:, 1]
    selected = [
        {"sentence_idx": i, "sentence": abstract[i], "probability": float(probs[i])}
        for i in range(len(abstract)) if probs[i] >= threshold
    ]
    selected.sort(key=lambda x: x["probability"], reverse=True)
    return selected

In [68]:
uploaded = files.upload()

Saving retrieved_docs.json to retrieved_docs.json


In [69]:
import json
with open("retrieved_docs.json") as f:
    retrieved = json.load(f)

In [70]:
def verify_claim(claim_text, claim_id, split="dev", top_k_docs=1):
    doc_ids = retrieved[split].get(str(claim_id), [])
    if not doc_ids:
        return {"verdict": "NOT_ENOUGH_INFO", "reason": ""}

    all_results = []

    for doc_id in doc_ids[:top_k_docs]:
        paper = corpus_by_id.get(doc_id)
        if paper is None:
            continue
        evidence_sentences = select_evidence(claim_text, paper, threshold=0.5)
        if not evidence_sentences:
            continue

        evidence_text = " ".join([e["sentence"] for e in evidence_sentences[:3]])
        result = predict_verdict(claim_text, evidence_text)
        result["doc_id"] = doc_id
        result["evidence_used"] = evidence_text
        all_results.append(result)

    if not all_results:
        return {"verdict": "NOT_ENOUGH_INFO", "reason": ""}
    best = max(all_results, key=lambda x: x["confidence"])
    return best

In [71]:
example_claim = next(c for c in claims_dev if c.get("evidence"))
result = verify_claim(example_claim["claim"], example_claim["id"], split="dev")
print("Claim:", example_claim["claim"])
print("Result:", result)

Claim: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
Result: {'verdict': 'SUPPORT', 'confidence': 0.6053385734558105, 'all_probs': {'SUPPORT': 0.6053385734558105, 'CONTRADICT': 0.07792835682630539, 'NOT_ENOUGH_INFO': 0.3167330324649811}, 'doc_id': 2739854, 'evidence_used': 'The fact that they tend not to identify more than a fraction of the specific causal loci has led to divergence of opinion over whether most of the variance is hidden as numerous rare variants of large effect or as common variants of very small effect.'}
